In [ ]:
import struct
import pandas as pd
import numpy as np
import os

def read_binary_features(feature_file_path, factor_names_file_path):
    """
    读取二进制特征文件并转换为DataFrame
    
    参数:
    - feature_file_path: 二进制特征文件路径
    - factor_names_file_path: 因子名称文件路径
    
    返回:
    - features_df: 包含所有特征的DataFrame
    """
    
    # 1. 读取因子名称
    print("读取因子名称...")
    factor_names_df = pd.read_csv(factor_names_file_path, header=None)
    factor_names = [name.replace('.csv', '') for name in factor_names_df.iloc[:, 0].tolist()]
    num_features = len(factor_names)
    print(f"因子数量: {num_features}")
    print(f"因子名称: {factor_names}")
    
    # 2. 读取二进制特征数据
    print(f"\n读取二进制特征文件: {feature_file_path}")
    
    if not os.path.exists(feature_file_path):
        raise FileNotFoundError(f"特征文件不存在: {feature_file_path}")
    
    with open(feature_file_path, 'rb') as f:
        binary_data = f.read()
    
    # 3. 解析二进制数据
    num_values = len(binary_data) // 8  # 每个double占8字节
    feature_values = struct.unpack(f'{num_values}d', binary_data)
    
    print(f"二进制文件大小: {len(binary_data)} 字节")
    print(f"解析出的数值数量: {num_values}")
    
    # 4. 计算时间点数量
    if num_values % num_features != 0:
        raise ValueError(f"数据不一致: 总数值数量 {num_values} 不能被特征数量 {num_features} 整除")
    
    num_timepoints = num_values // num_features
    print(f"时间点数量: {num_timepoints}")
    
    # 5. 重构DataFrame (从列优先转换为行优先)
    print(f"\n重构数据为DataFrame...")
    
    # 创建空的DataFrame
    features_df = pd.DataFrame(index=range(num_timepoints), columns=factor_names)
    
    # 按列优先的顺序填充数据
    for col_idx, col_name in enumerate(factor_names):
        start_idx = col_idx * num_timepoints
        end_idx = (col_idx + 1) * num_timepoints
        features_df[col_name] = feature_values[start_idx:end_idx]
    
    print(f"DataFrame形状: {features_df.shape}")
    
    return features_df

def display_feature_info(features_df, start_row=600, end_row=630):
    """
    显示特征文件的基本信息和指定行的数据
    
    参数:
    - features_df: 特征DataFrame
    - start_row: 开始行号
    - end_row: 结束行号
    """
    
    print("=" * 60)
    print("特征文件信息摘要")
    print("=" * 60)
    
    # 基本信息
    print(f"数据形状: {features_df.shape}")
    print(f"时间点数量: {features_df.shape[0]}")
    print(f"特征数量: {features_df.shape[1]}")
    
    # 列名信息
    print(f"\n列名 ({len(features_df.columns)} 个特征):")
    for i, col in enumerate(features_df.columns):
        print(f"  {i+1:2d}. {col}")
    
    # 数据统计
    print(f"\n数据统计信息:")
    print(f"数据类型: {features_df.dtypes.iloc[0]}")
    print(f"缺失值数量: {features_df.isnull().sum().sum()}")
    print(f"无穷值数量: {np.isinf(features_df.select_dtypes(include=[np.number])).sum().sum()}")
    
    # 数值范围
    numeric_df = features_df.select_dtypes(include=[np.number])
    if not numeric_df.empty:
        print(f"数值范围:")
        print(f"  最小值: {numeric_df.min().min():.6f}")
        print(f"  最大值: {numeric_df.max().max():.6f}")
        print(f"  均值: {numeric_df.mean().mean():.6f}")
    
    # 显示指定行的数据
    print(f"\n" + "=" * 60)
    print(f"第 {start_row} 到 {end_row} 行数据")
    print("=" * 60)
    
    if end_row > len(features_df):
        print(f"警告: 请求的结束行 {end_row} 超过了数据总行数 {len(features_df)}")
        end_row = len(features_df)
    
    if start_row >= len(features_df):
        print(f"错误: 请求的开始行 {start_row} 超过了数据总行数 {len(features_df)}")
        return
    
    # 显示数据
    display_data = features_df.iloc[start_row:end_row+1]
    
    print(f"显示行数: {len(display_data)}")
    print(f"实际行号范围: {start_row} 到 {start_row + len(display_data) - 1}")
    
    # 设置pandas显示选项
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.float_format', '{:.6f}'.format)
    
    print(f"\n数据内容:")
    print(display_data)
    
    # 重置pandas显示选项
    pd.reset_option('display.max_columns')
    pd.reset_option('display.width') 
    pd.reset_option('display.float_format')
    
    # 显示每列的统计信息（仅针对显示的行）
    print(f"\n第 {start_row}-{end_row} 行的统计信息:")
    stats = display_data.describe()
    print(stats)

def main():
    """
    主函数 - 读取并显示特征文件
    """
    
    # 设置文件路径
    feature_file_path = "/Users/wook/Downloads/FactorTest1/feature_test_data/feature/combined_features"
    factor_names_file_path = "/Users/wook/Downloads/FactorTest1/feature_error_test/factorname.csv"
    
    try:
        # 读取数据
        features_df = read_binary_features(feature_file_path, factor_names_file_path)
        
        # 显示信息
        display_feature_info(features_df, start_row=600, end_row=630)
        
        # 可选：保存为CSV文件便于查看
        save_csv = input(f"\n是否保存为CSV文件以便查看? (y/n): ").lower().strip()
        if save_csv == 'y':
            csv_path = feature_file_path + "_readable.csv"
            features_df.to_csv(csv_path, index=True)
            print(f"CSV文件已保存到: {csv_path}")
            
    except Exception as e:
        print(f"错误: {str(e)}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()